<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/TOPO2026_GLM_DEMO_CORRECTED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U bitsandbytes>=0.46.1 -q

## TOPO-2026

In [1]:
# ============================================================================
# TOPO-2026 FOR GLM-4.6 - CORRECT MODEL CLASS
# ============================================================================

"""
TOPO-2026 Implementation for GLM-4.6V-Flash (9B)
Uses correct model class for GLM-4.6
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import gc
import random
import hashlib
import time
import json
from typing import List, Dict, Tuple
from sklearn.metrics import accuracy_score
from transformers import (
    AutoModel,
    AutoTokenizer,
    AutoConfig,
    BitsAndBytesConfig
)
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("🔬 TOPO-2026: GLM-4.6V-Flash (CORRECT MODEL CLASS)")
print("="*80)

# ============================================================
# CONFIGURATION
# ============================================================
SEED          = 123
N_RUNS        = 3
BATCH_SIZE    = 8
MAX_LEN       = 64
EPOCHS        = 2
LR_EMBED      = 5e-3
LR_CLS        = 1e-3
PRIME_LIMIT   = 13
MODEL_NAME    = "zai-org/GLM-4.6V-Flash"
NUM_SAMPLES   = 500
EVAL_SIZE     = 200

print(f"\n📋 Configuration:")
print(f"   Model: {MODEL_NAME}")
print(f"   Runs: {N_RUNS}")
print(f"   Epochs: {EPOCHS}")
print(f"   Prime Limit: {PRIME_LIMIT}")

# ============================================================
# SETUP
# ============================================================
def set_seed(seed: int):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# ============================================================
# SYNTHETIC DATASET
# ============================================================
print("\n📚 Creating synthetic dataset...")

def create_synthetic_task(prefixes, num_samples=500):
    texts = []
    labels = []

    cat0_texts = [
        f"{prefixes[0]} {word}" for word in [
            "conference", "meeting", "summit", "agreement", "policy",
            "negotiation", "resolution", "sanctions", "diplomacy", "treaty",
            "security", "council", "alliance", "peace", "stability",
            "development", "mission", "delegation", "protocol", "mediation"
        ]
    ]

    cat1_texts = [
        f"{prefixes[1]} {word}" for word in [
            "tournament", "championship", "match", "league", "player",
            "coach", "stadium", "team", "victory", "defeat",
            "training", "season", "playoff", "referee", "penalty",
            "practice", "schedule", "competition", "score", "record"
        ]
    ]

    all_texts = cat0_texts + cat1_texts
    all_labels = [0] * len(cat0_texts) + [1] * len(cat1_texts)

    combined = list(zip(all_texts, all_labels))
    random.shuffle(combined)
    texts, labels = zip(*combined[:num_samples])

    return list(texts), list(labels)

task_a_texts, task_a_labels = create_synthetic_task(["World", "Sports"], NUM_SAMPLES)
task_b_texts, task_b_labels = create_synthetic_task(["Business", "Science"], NUM_SAMPLES)
task_c_texts, task_c_labels = create_synthetic_task(["Politics", "Technology"], NUM_SAMPLES)

print(f"   Task A: {len(task_a_texts)} samples (World vs Sports)")
print(f"   Task B: {len(task_b_texts)} samples (Business vs Science)")
print(f"   Task C: {len(task_c_texts)} samples (Politics vs Technology)")

# ============================================================
# PRIME KERNEL
# ============================================================
def primes_up_to(n):
    sieve = [True] * (n + 1)
    sieve[0] = sieve[1] = False
    for i in range(2, int(n ** 0.5) + 1):
        if sieve[i]:
            for j in range(i * i, n + 1, i):
                sieve[j] = False
    return [i for i in range(2, n + 1) if sieve[i]]

PRIME_ANCHORS = primes_up_to(PRIME_LIMIT)
SAFETY_CONSTANT = 1.0 - np.prod([1.0 - (p ** -0.5) for p in PRIME_ANCHORS])

print(f"\n🔢 Prime Anchors: {PRIME_ANCHORS}")
print(f"🔒 Safety Constant Λ: {SAFETY_CONSTANT:.10f}")

# ============================================================
# MODEL ARCHITECTURE (Using AutoModel, not AutoModelForCausalLM)
# ============================================================
class TaskAwareGLM(nn.Module):
    def __init__(self, base_model, hidden_size=4096):
        super().__init__()
        self.base_model = base_model
        self.classifier_A = nn.Linear(hidden_size, 2, dtype=torch.bfloat16)
        self.classifier_B = nn.Linear(hidden_size, 2, dtype=torch.bfloat16)
        self.classifier_C = nn.Linear(hidden_size, 2, dtype=torch.bfloat16)
        self.current_task = 'A'

    def forward(self, input_ids, attention_mask=None):
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )
        # Get last hidden state
        if hasattr(outputs, 'last_hidden_state'):
            last_hidden = outputs.last_hidden_state[:, -1, :]
        else:
            last_hidden = outputs.hidden_states[-1][:, -1, :]

        head = getattr(self, f'classifier_{self.current_task}')
        return head(last_hidden)

    def switch_task(self, task: str):
        assert task in ('A', 'B', 'C')
        self.current_task = task

# ============================================================
# UTILITIES
# ============================================================
def get_embed_layer(base_model):
    """Find embedding layer in GLM-4.6"""
    # Try common locations
    if hasattr(base_model, 'transformer') and hasattr(base_model.transformer, 'word_embeddings'):
        return base_model.transformer.word_embeddings
    if hasattr(base_model, 'model') and hasattr(base_model.model, 'embed_tokens'):
        return base_model.model.embed_tokens
    if hasattr(base_model, 'embed_tokens'):
        return base_model.embed_tokens
    if hasattr(base_model, 'word_embeddings'):
        return base_model.word_embeddings

    # Search for embedding layer
    for module in base_model.modules():
        if isinstance(module, nn.Embedding) and module.weight.shape[0] > 100_000:
            return module

    raise RuntimeError("Could not locate embedding layer.")

@torch.no_grad()
def evaluate(model, tokenizer, texts, labels, task: str, batch_size: int = 32) -> float:
    was_training = model.training
    previous_task = model.current_task
    model.eval()
    model.switch_task(task)

    all_preds, all_labels = [], []
    for i in range(0, len(texts), batch_size):
        tokens = tokenizer(
            texts[i:i + batch_size], return_tensors="pt",
            padding=True, truncation=True, max_length=MAX_LEN,
        ).to(device)
        logits = model(tokens.input_ids, tokens.attention_mask)
        all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        all_labels.extend(labels[i:i + batch_size])

    model.switch_task(previous_task)
    if was_training:
        model.train()
    return accuracy_score(all_labels, all_preds)

def tokenize(tokenizer, texts, labels):
    tokens = tokenizer(
        texts, return_tensors="pt",
        padding=True, truncation=True, max_length=MAX_LEN,
    ).to(device)
    return tokens, torch.tensor(labels, dtype=torch.long).to(device)

def flush_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

def cleanup(*objects):
    for obj in objects:
        del obj
    gc.collect()
    torch.cuda.empty_cache()

# ============================================================
# TOPOLOGICAL GOVERNOR
# ============================================================
class TopologicalGovernor:
    def __init__(self, embed_layer: nn.Embedding):
        self.embed_layer = embed_layer
        vocab_size = embed_layer.weight.shape[0]
        self.anchor_indices = [p for p in PRIME_ANCHORS if p < vocab_size]
        self.snapshot = {}
        self.safety_constant = SAFETY_CONSTANT

    def take_snapshot(self):
        self.snapshot = {
            idx: self.embed_layer.weight[idx].detach().clone().float()
            for idx in self.anchor_indices
        }

    @torch.no_grad()
    def enforce_anchors(self):
        if not self.snapshot:
            raise RuntimeError("Call take_snapshot() first.")
        dtype = self.embed_layer.weight.dtype
        for idx, cached in self.snapshot.items():
            self.embed_layer.weight[idx].copy_(cached.to(dtype=dtype))

    @torch.no_grad()
    def zero_anchor_gradients(self):
        if self.embed_layer.weight.grad is not None:
            for idx in self.anchor_indices:
                self.embed_layer.weight.grad[idx].zero_()

    def verify_integrity(self, atol: float = 1e-5) -> bool:
        return all(
            torch.allclose(self.embed_layer.weight[idx].float(), cached, atol=atol)
            for idx, cached in self.snapshot.items()
        )

# ============================================================
# TRAINING FUNCTION
# ============================================================
def train_topo_glm46():
    print(f"\n{'='*60}")
    print("🚀 TOPO-2026 Training on GLM-4.6")
    print(f"{'='*60}")

    all_results = []
    final_model = None
    final_tokenizer = None

    for run in range(N_RUNS):
        set_seed(SEED + run)
        print(f"\n  📍 Run {run + 1}/{N_RUNS}")

        # Load GLM-4.6 with 4-bit quantization using AutoModel
        print("  Loading GLM-4.6V-Flash (4-bit)...")

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16
        )

        # Use AutoModel instead of AutoModelForCausalLM
        base_model = AutoModel.from_pretrained(
            MODEL_NAME,
            trust_remote_code=True,
            quantization_config=bnb_config,
            device_map="auto",
            torch_dtype=torch.bfloat16
        )

        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        # Build task-aware model
        model = TaskAwareGLM(base_model).to(device)
        embed_layer = get_embed_layer(base_model)
        embed_layer.weight.requires_grad = True

        # Freeze everything except embedding and classifiers
        for name, param in base_model.named_parameters():
            if 'embed' not in name.lower() and 'wte' not in name.lower():
                param.requires_grad = False

        # Optimizer
        opt = torch.optim.AdamW([
            {'params': embed_layer.weight, 'lr': LR_EMBED},
            {'params': model.classifier_A.parameters(), 'lr': LR_CLS},
            {'params': model.classifier_B.parameters(), 'lr': LR_CLS},
            {'params': model.classifier_C.parameters(), 'lr': LR_CLS},
        ])

        # --- TASK A ---
        print("  📚 Task A (World vs Sports)")
        model.switch_task('A')
        model.train()
        for epoch in range(EPOCHS):
            epoch_loss = 0
            for i in range(0, len(task_a_texts), BATCH_SIZE):
                tokens, labels = tokenize(
                    tokenizer,
                    task_a_texts[i:i + BATCH_SIZE],
                    task_a_labels[i:i + BATCH_SIZE],
                )
                opt.zero_grad()
                logits = model(tokens.input_ids, tokens.attention_mask)
                loss = F.cross_entropy(logits, labels)
                loss.backward()
                opt.step()
                epoch_loss += loss.item()
            print(f"    Epoch {epoch+1}/{EPOCHS}: Loss={epoch_loss/len(task_a_texts)*BATCH_SIZE:.4f}")

        # --- SNAPSHOT ---
        governor = TopologicalGovernor(embed_layer)
        t0 = time.perf_counter()
        governor.take_snapshot()
        snap_time = (time.perf_counter() - t0) * 1000
        anchor_mem = (len(governor.anchor_indices) * embed_layer.weight.shape[1] * 4) / 1024

        print(f"  🔒 Anchored {len(governor.anchor_indices)} prime rows: {governor.anchor_indices}")
        print(f"  ⏱️  Snapshot: {snap_time:.2f}ms | Memory: {anchor_mem:.2f}KB")

        model.classifier_A.requires_grad_(False)
        acc_a0 = evaluate(model, tokenizer, task_a_texts[:EVAL_SIZE], task_a_labels[:EVAL_SIZE], 'A')

        # --- TASK B ---
        print("  📚 Task B (Business vs Science)")
        model.switch_task('B')
        model.train()
        opt_b = torch.optim.AdamW([
            {'params': embed_layer.weight, 'lr': LR_EMBED},
            {'params': model.classifier_B.parameters(), 'lr': LR_CLS},
        ])

        for epoch in range(EPOCHS):
            epoch_loss = 0
            for i in range(0, len(task_b_texts), BATCH_SIZE):
                tokens, labels = tokenize(
                    tokenizer,
                    task_b_texts[i:i + BATCH_SIZE],
                    task_b_labels[i:i + BATCH_SIZE],
                )
                opt_b.zero_grad()
                logits = model(tokens.input_ids, tokens.attention_mask)
                loss = F.cross_entropy(logits, labels)
                loss.backward()
                governor.zero_anchor_gradients()
                torch.nn.utils.clip_grad_norm_(embed_layer.weight, max_norm=1.0)
                opt_b.step()
                governor.enforce_anchors()
                epoch_loss += loss.item()
            print(f"    Epoch {epoch+1}/{EPOCHS}: Loss={epoch_loss/len(task_b_texts)*BATCH_SIZE:.4f}")

        model.classifier_B.requires_grad_(False)
        acc_b0 = evaluate(model, tokenizer, task_b_texts[:EVAL_SIZE], task_b_labels[:EVAL_SIZE], 'B')

        # --- TASK C ---
        print("  📚 Task C (Politics vs Technology)")
        model.switch_task('C')
        model.train()
        opt_c = torch.optim.AdamW([
            {'params': embed_layer.weight, 'lr': LR_EMBED},
            {'params': model.classifier_C.parameters(), 'lr': LR_CLS},
        ])

        for epoch in range(EPOCHS):
            epoch_loss = 0
            for i in range(0, len(task_c_texts), BATCH_SIZE):
                tokens, labels = tokenize(
                    tokenizer,
                    task_c_texts[i:i + BATCH_SIZE],
                    task_c_labels[i:i + BATCH_SIZE],
                )
                opt_c.zero_grad()
                logits = model(tokens.input_ids, tokens.attention_mask)
                loss = F.cross_entropy(logits, labels)
                loss.backward()
                governor.zero_anchor_gradients()
                torch.nn.utils.clip_grad_norm_(embed_layer.weight, max_norm=1.0)
                opt_c.step()
                governor.enforce_anchors()
                epoch_loss += loss.item()
            print(f"    Epoch {epoch+1}/{EPOCHS}: Loss={epoch_loss/len(task_c_texts)*BATCH_SIZE:.4f}")

        assert governor.verify_integrity(), "❌ Anchor integrity FAILED!"

        # --- EVALUATE ---
        acc_a1 = evaluate(model, tokenizer, task_a_texts[:EVAL_SIZE], task_a_labels[:EVAL_SIZE], 'A')
        acc_b1 = evaluate(model, tokenizer, task_b_texts[:EVAL_SIZE], task_b_labels[:EVAL_SIZE], 'B')
        acc_c = evaluate(model, tokenizer, task_c_texts[:EVAL_SIZE], task_c_labels[:EVAL_SIZE], 'C')

        forget_a = (acc_a0 - acc_a1) * 100
        forget_b = (acc_b0 - acc_b1) * 100
        forget_comb = (forget_a + forget_b) / 2

        print(f"\n  📊 Results:")
        print(f"    Task A: {acc_a0*100:.1f}% → {acc_a1*100:.1f}% | Forgetting: {forget_a:+.1f}%")
        print(f"    Task B: {acc_b0*100:.1f}% → {acc_b1*100:.1f}% | Forgetting: {forget_b:+.1f}%")
        print(f"    Combined Forgetting: {forget_comb:+.1f}%")
        print(f"    Task C Accuracy: {acc_c*100:.1f}%")

        all_results.append({
            'forget_a': forget_a,
            'forget_b': forget_b,
            'forget_comb': forget_comb,
            'acc_c': acc_c,
            'acc_a1': acc_a1,
            'acc_b1': acc_b1,
            'snap_time_ms': snap_time,
            'anchor_mem_kb': anchor_mem,
        })

        if run == N_RUNS - 1:
            final_model = model
            final_tokenizer = tokenizer
            cleanup(base_model)
        else:
            cleanup(model, base_model)

        flush_gpu()

    return all_results, final_model, final_tokenizer

# ============================================================
# RUN TRAINING
# ============================================================
print("\n" + "="*80)
print("🚀 STARTING TOPO-2026 TRAINING")
print("="*80)

results, final_model, final_tokenizer = train_topo_glm46()

# ============================================================
# RESULTS SUMMARY
# ============================================================
print("\n" + "="*80)
print("📊 TOPO-2026 RESULTS SUMMARY")
print("="*80)

forget_a_vals = [r['forget_a'] for r in results]
forget_b_vals = [r['forget_b'] for r in results]
forget_comb_vals = [r['forget_comb'] for r in results]
acc_c_vals = [r['acc_c'] for r in results]

print(f"\n{'Metric':<35} {'Value':<20}")
print("-"*55)
print(f"{'Task C Accuracy':<35} {np.mean(acc_c_vals)*100:>5.1f}% ±{np.std(acc_c_vals)*100:>4.1f}")
print(f"{'Combined Forgetting':<35} {np.mean(forget_comb_vals):>+5.1f}% ±{np.std(forget_comb_vals):>4.1f}")
print(f"{'Forgetting A':<35} {np.mean(forget_a_vals):>+5.1f}% ±{np.std(forget_a_vals):>4.1f}")
print(f"{'Forgetting B':<35} {np.mean(forget_b_vals):>+5.1f}% ±{np.std(forget_b_vals):>4.1f}")
print(f"{'Snapshot Time':<35} {np.mean([r['snap_time_ms'] for r in results]):>5.2f}ms")
print(f"{'Anchor Memory':<35} {np.mean([r['anchor_mem_kb'] for r in results]):>5.2f}KB")

# ============================================================
# CERTIFICATION BADGE
# ============================================================
task_c_pass = "PASS" if np.mean(acc_c_vals) >= 0.95 else "FAIL"
forget_pass = "PASS" if np.mean(forget_comb_vals) <= 10.0 else "FAIL"

print("\n" + "="*80)
print("🔬 TOPO-2026 CERTIFICATION")
print("="*80)

print(f"""
+------------------------------------------+
| TOPOLOGICAL AI CERTIFIED                 |
| |- Task C Accuracy: {np.mean(acc_c_vals)*100:.1f}% (>=95%) {task_c_pass:>4} |
| |- Combined Forgetting: {np.mean(forget_comb_vals):.1f}% (<=10%) {forget_pass:>4} |
| |- Anchor Integrity:              PASS |
| |- Runs Completed: {N_RUNS}/{N_RUNS}              PASS |
| `- Standard: TOPO-2026                   |
+------------------------------------------+
""")

# ============================================================
# SAVE MODEL  (CORRECTED)
# ============================================================
# The base model was loaded in 4-bit (bitsandbytes). Calling
# .state_dict() on it dumps PACKED nf4 blobs + quant metadata, which
# cannot be reloaded into a normal model -- that was the original bug.
#
# What actually changed during training is tiny: the 3 classifier heads
# and the trained embedding rows. We save ONLY those, as plain float
# tensors, so they reload cleanly on top of a fresh stock GLM base.
if final_model:
    print("\n💾 Saving trained parts (correct, reloadable)...")
    embed_w = get_embed_layer(final_model.base_model).weight.detach().cpu().float()
    torch.save(
        {
            "classifier_A": {k: v.cpu() for k, v in final_model.classifier_A.state_dict().items()},
            "classifier_B": {k: v.cpu() for k, v in final_model.classifier_B.state_dict().items()},
            "classifier_C": {k: v.cpu() for k, v in final_model.classifier_C.state_dict().items()},
            "embed_tokens_weight": embed_w,          # full embedding incl. trained rows
            "prime_anchors": PRIME_ANCHORS,
            "safety_constant": float(SAFETY_CONSTANT),
            "hidden_size": 4096,
            "base_model": MODEL_NAME,
            "max_len": MAX_LEN,
        },
        "topo_trained_parts.pt",
    )
    print("✅ Saved: topo_trained_parts.pt  (classifiers + embeddings + anchors)")

    if final_tokenizer:
        final_tokenizer.save_pretrained("./glm46_topo")
        print("✅ Tokenizer saved: ./glm46_topo")

# ============================================================
# CERTIFICATION DATA
# ============================================================
cert_data = {
    "model": "GLM-4.6V-Flash",
    "base_model": MODEL_NAME,
    "certification_standard": "TOPO-2026",
    "certification_status": "CERTIFIED" if (task_c_pass == "PASS" and forget_pass == "PASS") else "NOT CERTIFIED",
    "prime_anchors": PRIME_ANCHORS,
    "safety_constant": SAFETY_CONSTANT,
    "runs": N_RUNS,
    "seed": SEED,
    "results": {
        "task_c_accuracy": f"{np.mean(acc_c_vals)*100:.1f}%",
        "combined_forgetting": f"{np.mean(forget_comb_vals):.1f}%",
        "anchor_memory_kb": f"{np.mean([r['anchor_mem_kb'] for r in results]):.2f}",
        "snapshot_time_ms": f"{np.mean([r['snap_time_ms'] for r in results]):.2f}"
    }
}

with open("topo_certification.json", "w") as f:
    json.dump(cert_data, f, indent=2)
print("✅ Certification data saved: topo_certification.json")

print("\n" + "="*80)
print("🎉 TOPO-2026 TRAINING COMPLETE!")
print("="*80)

print(f"""
📊 CERTIFICATION SUMMARY:
   Standard: TOPO-2026
   Status: {"✅ CERTIFIED" if task_c_pass == "PASS" and forget_pass == "PASS" else "❌ NOT CERTIFIED"}
   Task C Accuracy: {np.mean(acc_c_vals)*100:.1f}%
   Combined Forgetting: {np.mean(forget_comb_vals):.1f}%
   Prime Anchors: {PRIME_ANCHORS}
   Safety Constant: {SAFETY_CONSTANT:.10f}

📁 SAVED FILES:
   ✅ glm-4.6-topo-2026.pt (Model weights)
   ✅ topo_certification.json (Certification)
   ✅ ./glm46_topo/ (Tokenizer)
""")

print("="*80)

🔬 TOPO-2026: GLM-4.6V-Flash (CORRECT MODEL CLASS)

📋 Configuration:
   Model: zai-org/GLM-4.6V-Flash
   Runs: 3
   Epochs: 2
   Prime Limit: 13
✅ Device: cuda
   GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
   VRAM: 95.0 GB

📚 Creating synthetic dataset...
   Task A: 40 samples (World vs Sports)
   Task B: 40 samples (Business vs Science)
   Task C: 40 samples (Politics vs Technology)

🔢 Prime Anchors: [2, 3, 5, 7, 11, 13]
🔒 Safety Constant Λ: 0.9785142874

🚀 STARTING TOPO-2026 TRAINING

🚀 TOPO-2026 Training on GLM-4.6

  📍 Run 1/3
  Loading GLM-4.6V-Flash (4-bit)...


config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/66.4k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

[transformers] Glm4vModel LOAD REPORT from: zai-org/GLM-4.6V-Flash
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  📚 Task A (World vs Sports)
    Epoch 1/2: Loss=1.0276
    Epoch 2/2: Loss=0.0280
  🔒 Anchored 6 prime rows: [2, 3, 5, 7, 11, 13]
  ⏱️  Snapshot: 0.24ms | Memory: 96.00KB
  📚 Task B (Business vs Science)
    Epoch 1/2: Loss=2.5070
    Epoch 2/2: Loss=0.0033
  📚 Task C (Politics vs Technology)
    Epoch 1/2: Loss=1.4322
    Epoch 2/2: Loss=0.0250

  📊 Results:
    Task A: 97.5% → 100.0% | Forgetting: -2.5%
    Task B: 100.0% → 97.5% | Forgetting: +2.5%
    Combined Forgetting: +0.0%
    Task C Accuracy: 97.5%

  📍 Run 3/3
  Loading GLM-4.6V-Flash (4-bit)...


Loading weights:   0%|          | 0/703 [00:00<?, ?it/s]

[transformers] Glm4vModel LOAD REPORT from: zai-org/GLM-4.6V-Flash
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  📚 Task A (World vs Sports)
    Epoch 1/2: Loss=0.9059
    Epoch 2/2: Loss=0.2346
  🔒 Anchored 6 prime rows: [2, 3, 5, 7, 11, 13]
  ⏱️  Snapshot: 0.10ms | Memory: 96.00KB
  📚 Task B (Business vs Science)
    Epoch 1/2: Loss=1.3199
    Epoch 2/2: Loss=0.1469
  📚 Task C (Politics vs Technology)
    Epoch 1/2: Loss=0.3176
    Epoch 2/2: Loss=0.1687

  📊 Results:
    Task A: 100.0% → 95.0% | Forgetting: +5.0%
    Task B: 100.0% → 97.5% | Forgetting: +2.5%
    Combined Forgetting: +3.8%
    Task C Accuracy: 97.5%

📊 TOPO-2026 RESULTS SUMMARY

Metric                              Value               
-------------------------------------------------------
Task C Accuracy                      97.5% ± 0.0
Combined Forgetting                  +2.1% ± 1.6
Forgetting A                         +2.5% ± 3.5
Forgetting B                         +1.7% ± 1.2
Snapshot Time                        0.18ms
Anchor Memory                       96.00KB

🔬 TOPO-2026 CERTIFICATION

+-----------------------------

## HF

In [2]:
# ============================================================================
# DEPLOY TOPO-2026 TRAINED PARTS TO HF  (CORRECTED)
# ============================================================================
# Uploads the reloadable checkpoint produced by the corrected save step:
#   topo_trained_parts.pt  + tokenizer + a config that documents the setup.
# NOTE: this is a CLASSIFIER checkpoint (3 task heads + anchored embeddings),
# not a standalone generative model. It is meant to be reloaded on top of a
# fresh 4-bit GLM base, exactly as in training.

import os, shutil, json
from huggingface_hub import HfApi, create_repo, upload_folder, login
try:
    from google.colab import userdata
    _HF = userdata.get("HF_TOKEN")
except Exception:
    _HF = os.environ.get("HF_TOKEN")

REPO_ID        = "frankmorales2020/glm-4.6-topo-2026"
LOCAL_PATH     = "./glm46_deploy"
TRAINED_PARTS  = "topo_trained_parts.pt"
TOKENIZER_PATH = "./glm46_topo"

print("="*80); print("📤 DEPLOY (corrected)"); print("="*80)
print(f"Repository: {REPO_ID}")

HF_TOKEN = _HF or input("Enter your HF token: ")
login(token=HF_TOKEN, add_to_git_credential=True)
api = HfApi()
print("👤 User:", api.whoami(token=HF_TOKEN)["name"])

os.makedirs(LOCAL_PATH, exist_ok=True)

if not os.path.exists(TRAINED_PARTS):
    raise FileNotFoundError(f"{TRAINED_PARTS} not found. Run the training cell first.")
shutil.copy(TRAINED_PARTS, f"{LOCAL_PATH}/{TRAINED_PARTS}")
print(f"  ✅ {TRAINED_PARTS} ({os.path.getsize(TRAINED_PARTS)/1024**2:.1f} MB)")

if os.path.exists(TOKENIZER_PATH):
    for f in os.listdir(TOKENIZER_PATH):
        shutil.copy(f"{TOKENIZER_PATH}/{f}", f"{LOCAL_PATH}/{f}")
    print("  ✅ tokenizer files")

# A descriptive config (NOT a transformers model config -- this isn't a
# standalone model). Records exactly how to reload it.
meta = {
    "artifact_type": "topo2026_task_classifier",
    "base_model": "zai-org/GLM-4.6V-Flash",
    "load_base_in_4bit": True,
    "components": ["classifier_A", "classifier_B", "classifier_C",
                   "embed_tokens_weight", "prime_anchors", "safety_constant"],
    "tasks": {"A": "World vs Sports", "B": "Business vs Science",
              "C": "Politics vs Technology"},
    "note": "Reload with the inference cell: load stock GLM base in 4-bit, "
            "rebuild TaskAwareGLM, load topo_trained_parts.pt."
}
json.dump(meta, open(f"{LOCAL_PATH}/topo_config.json","w"), indent=2)
print("  ✅ topo_config.json")

with open(f"{LOCAL_PATH}/.gitattributes","w") as f:
    f.write("*.pt filter=lfs diff=lfs merge=lfs -text\n")

create_repo(repo_id=REPO_ID, token=HF_TOKEN, private=False,
            repo_type="model", exist_ok=True)
upload_folder(repo_id=REPO_ID, folder_path=LOCAL_PATH, repo_type="model",
              commit_message="TOPO-2026 trained parts (corrected, reloadable)")
print("✅ Upload complete:", f"https://huggingface.co/{REPO_ID}")


📤 DEPLOY (corrected)
Repository: frankmorales2020/glm-4.6-topo-2026
👤 User: frankmorales2020
  ✅ topo_trained_parts.pt (2368.1 MB)
  ✅ tokenizer files
  ✅ topo_config.json


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...m46_deploy/tokenizer.json: 100%|##########| 20.0MB / 20.0MB            

  ...loy/topo_trained_parts.pt:  12%|#1        |  292MB / 2.48GB            

✅ Upload complete: https://huggingface.co/frankmorales2020/glm-4.6-topo-2026


## INFERENCE  (corrected: classification, not chat)

In [3]:
# ============================================================================
# DIAGNOSTIC: inspect the corrected checkpoint
# ============================================================================
import torch
from huggingface_hub import hf_hub_download

REPO_ID = "frankmorales2020/glm-4.6-topo-2026"
path = hf_hub_download(REPO_ID, "topo_trained_parts.pt")
ckpt = torch.load(path, map_location="cpu", weights_only=False)

print("Keys in checkpoint:")
for k, v in ckpt.items():
    if hasattr(v, "shape"):
        print(f"  {k:24s} tensor {tuple(v.shape)} {v.dtype}")
    elif isinstance(v, dict):
        print(f"  {k:24s} dict -> {[ (kk, tuple(vv.shape)) for kk,vv in v.items()]}")
    else:
        print(f"  {k:24s} {v}")


topo_trained_parts.pt:   0%|          | 0.00/2.48G [00:00<?, ?B/s]

Keys in checkpoint:
  classifier_A             dict -> [('weight', (2, 4096)), ('bias', (2,))]
  classifier_B             dict -> [('weight', (2, 4096)), ('bias', (2,))]
  classifier_C             dict -> [('weight', (2, 4096)), ('bias', (2,))]
  embed_tokens_weight      tensor (151552, 4096) torch.float32
  prime_anchors            [2, 3, 5, 7, 11, 13]
  safety_constant          0.9785142874363926
  hidden_size              4096
  base_model               zai-org/GLM-4.6V-Flash
  max_len                  64


In [4]:
# ============================================================================
# INFERENCE (CORRECTED): reload the TOPO-2026 classifier and run predictions
# ============================================================================
# IMPORTANT: what was trained is a 3-task TEXT CLASSIFIER on top of a FROZEN
# GLM base. It cannot "chat" / generate free text -- that capability was never
# trained. So the correct test is classification, which is what this cell does.
#
# Requires the same classes/utilities defined in the training cell:
#   TaskAwareGLM, get_embed_layer, MODEL_NAME, BitsAndBytesConfig, etc.
# Run the training cell (or at least its class/import definitions) first.

import torch
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub import hf_hub_download

REPO_ID = "frankmorales2020/glm-4.6-topo-2026"
device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("📥 Loading stock GLM base in 4-bit (same as training)...")
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True,
                         bnb_4bit_compute_dtype=torch.bfloat16)
base_model = AutoModel.from_pretrained(
    MODEL_NAME, trust_remote_code=True, quantization_config=bnb,
    device_map="auto", torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("📥 Rebuilding TaskAwareGLM and loading trained parts...")
model = TaskAwareGLM(base_model).to(device)

ckpt = torch.load(hf_hub_download(REPO_ID, "topo_trained_parts.pt"),
                  map_location="cpu", weights_only=False)
model.classifier_A.load_state_dict({k: v.to(model.classifier_A.weight.dtype)
                                    for k, v in ckpt["classifier_A"].items()})
model.classifier_B.load_state_dict({k: v.to(model.classifier_B.weight.dtype)
                                    for k, v in ckpt["classifier_B"].items()})
model.classifier_C.load_state_dict({k: v.to(model.classifier_C.weight.dtype)
                                    for k, v in ckpt["classifier_C"].items()})

# Restore trained embedding rows (copy into the live embedding table)
with torch.no_grad():
    emb = get_embed_layer(base_model)
    w = ckpt["embed_tokens_weight"].to(emb.weight.dtype).to(emb.weight.device)
    if w.shape == emb.weight.shape:
        emb.weight.copy_(w)
        print("  ✅ embedding weights restored")
    else:
        print(f"  ⚠️ embedding shape mismatch {tuple(w.shape)} vs {tuple(emb.weight.shape)}; skipped")
model.eval()
print("✅ Classifier ready. Anchors:", ckpt["prime_anchors"],
      "| Λ =", round(ckpt["safety_constant"], 6))

TASK_LABELS = {
    "A": ["World", "Sports"],
    "B": ["Business", "Science"],
    "C": ["Politics", "Technology"],
}

@torch.no_grad()
def classify(text, task):
    model.switch_task(task)
    tok = tokenizer([text], return_tensors="pt", padding=True,
                    truncation=True, max_length=ckpt.get("max_len", 64)).to(device)
    logits = model(tok.input_ids, tok.attention_mask)
    probs = torch.softmax(logits, dim=1)[0]
    idx = int(torch.argmax(probs))
    return TASK_LABELS[task][idx], float(probs[idx])

print("\n" + "="*80 + "\n🚀 CLASSIFICATION TESTS\n" + "="*80)
tests = [
    ("World summit on climate policy",        "A"),
    ("The team won the championship match",   "A"),
    ("Quarterly business earnings report",    "B"),
    ("New scientific research breakthrough",  "B"),
    ("Government election and policy debate", "C"),
    ("Latest smartphone technology release",  "C"),
]
for text, task in tests:
    label, conf = classify(text, task)
    print(f"[{task}] {text!r:45s} -> {label:12s} ({conf*100:.1f}%)")

print("\n" + "="*80 + "\n🏁 INFERENCE COMPLETE\n" + "="*80)


📥 Loading stock GLM base in 4-bit (same as training)...


Loading weights:   0%|          | 0/703 [00:00<?, ?it/s]

[transformers] Glm4vModel LOAD REPORT from: zai-org/GLM-4.6V-Flash
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


📥 Rebuilding TaskAwareGLM and loading trained parts...
  ✅ embedding weights restored
✅ Classifier ready. Anchors: [2, 3, 5, 7, 11, 13] | Λ = 0.978514

🚀 CLASSIFICATION TESTS
[A] 'World summit on climate policy'              -> World        (100.0%)
[A] 'The team won the championship match'         -> Sports       (100.0%)
[B] 'Quarterly business earnings report'          -> Business     (100.0%)
[B] 'New scientific research breakthrough'        -> Science      (98.8%)
[C] 'Government election and policy debate'       -> Politics     (100.0%)
[C] 'Latest smartphone technology release'        -> Technology   (100.0%)

🏁 INFERENCE COMPLETE
